In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path_train = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_testing-set.csv"
df_train = pd.read_csv(path_train)

path_test = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_training-set.csv"
df_test = pd.read_csv(path_test)

In [ ]:
print(df_train.shape)
print(df_test.shape)

(175341, 45)
(82332, 45)


In [ ]:
df = pd.concat([df_train, df_test], axis=0)
# df = df_test
df = df.sample(frac=1)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df['label'].value_counts()

,count
label,
1,164673
0,93000


In [ ]:
df = df.groupby('label')
df = df.sample(n=df.size().min(), replace=True)
df = df.reset_index(drop=True)

In [ ]:
df['label'].value_counts()

,count
label,
0,93000
1,93000


In [ ]:
# list = [i for i in range(11) if i % 2 == 0]
df_columns = df.columns
df_num_columns = df.describe().columns
df_cat_columns = [col for col in df_columns if col not in df_num_columns]

In [ ]:
df_cat_columns

['proto', 'service', 'state', 'attack_cat']

In [ ]:
# df.proto.value_counts()

In [ ]:
def labelEncoder(dataframe, col_name_list):
    from sklearn import preprocessing
    label_encoder = preprocessing.LabelEncoder()
    encoding_dict = {}

    for col_name in col_name_list:
        dataframe[col_name] = label_encoder.fit_transform(dataframe[col_name])
        # Store the mapping for this column
        encoding_dict[col_name] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    return dataframe, encoding_dict

# Apply the function to your datasets
df, encoding_dict_train = labelEncoder(df, df_cat_columns)
df_test, encoding_dict_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
encoding_dict_train

{'proto': {'3pc': np.int64(0),
  'a/n': np.int64(1),
  'aes-sp3-d': np.int64(2),
  'any': np.int64(3),
  'argus': np.int64(4),
  'aris': np.int64(5),
  'arp': np.int64(6),
  'ax.25': np.int64(7),
  'bbn-rcc': np.int64(8),
  'bna': np.int64(9),
  'br-sat-mon': np.int64(10),
  'cbt': np.int64(11),
  'cftp': np.int64(12),
  'chaos': np.int64(13),
  'compaq-peer': np.int64(14),
  'cphb': np.int64(15),
  'cpnx': np.int64(16),
  'crtp': np.int64(17),
  'crudp': np.int64(18),
  'dcn': np.int64(19),
  'ddp': np.int64(20),
  'ddx': np.int64(21),
  'dgp': np.int64(22),
  'egp': np.int64(23),
  'eigrp': np.int64(24),
  'emcon': np.int64(25),
  'encap': np.int64(26),
  'etherip': np.int64(27),
  'fc': np.int64(28),
  'fire': np.int64(29),
  'ggp': np.int64(30),
  'gmtp': np.int64(31),
  'gre': np.int64(32),
  'hmp': np.int64(33),
  'i-nlsp': np.int64(34),
  'iatp': np.int64(35),
  'ib': np.int64(36),
  'icmp': np.int64(37),
  'idpr': np.int64(38),
  'idpr-cmtp': np.int64(39),
  'idrp': np.int64(40

In [ ]:
def labelEncoder(dataframe, col_name_list):
  from sklearn import preprocessing
  label_encoder = preprocessing.LabelEncoder()

  for col_name in col_name_list:
    dataframe[col_name]= label_encoder.fit_transform(dataframe[col_name])

  return dataframe

df = labelEncoder(df, df_cat_columns)
df_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
X = df.drop(['attack_cat', 'label'], axis = 1)
y = df['label']
y_mul = df['attack_cat']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, shuffle= True, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle= True, stratify=y_test)

In [ ]:
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)

In [ ]:
y_train

,label
0,0
1,0
2,1
3,0
4,0
...,...
148795,0
148796,0
148797,1
148798,1


# 100% Data (Supervised model)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(random_state=0)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
# y_pred = model.predict(X_val)
# accuracy = accuracy_score(y_val, y_pred)
# print(accuracy)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

classification_report = classification_report(y_test, y_pred, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred)

# print("Accuracy:", accuracy)
print("Classification report: \n", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

Classification report: 
               precision    recall  f1-score   support

           0     0.9850    0.9915    0.9883      9300
           1     0.9914    0.9849    0.9882      9300

    accuracy                         0.9882     18600
   macro avg     0.9882    0.9882    0.9882     18600
weighted avg     0.9882    0.9882    0.9882     18600

Confusion matrix: [[9221   79]
 [ 140 9160]]
AUC score: 0.9882258064516127


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

y_pred = model.predict(X_val)
classification_report = classification_report(y_val, y_pred, digits=4)
confusion_matrix = confusion_matrix(y_val, y_pred)
auc_score = roc_auc_score(y_val, y_pred)

# print("Accuracy:", accuracy)
print("Classification report: \n", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

Classification report: 
               precision    recall  f1-score   support

           0     0.9850    0.9929    0.9889      9300
           1     0.9928    0.9848    0.9888      9300

    accuracy                         0.9889     18600
   macro avg     0.9889    0.9889    0.9889     18600
weighted avg     0.9889    0.9889    0.9889     18600

Confusion matrix: [[9234   66]
 [ 141 9159]]
AUC score: 0.9888709677419355


**20% Labeled Data**

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Split feature set into two random views
from sklearn.model_selection import ShuffleSplit

splitter = ShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in splitter.split(X_train.T):
    view1_index, view2_index = train_index, test_index

X_train_view1 = X_train.iloc[:, view1_index]
X_train_view2 = X_train.iloc[:, view2_index]
X_val_view1 = X_val.iloc[:, view1_index]
X_val_view2 = X_val.iloc[:, view2_index]
X_test_view1 = X_test.iloc[:, view1_index]
X_test_view2 = X_test.iloc[:, view2_index]


# Split labeled and unlabeled data (20% labeled)
labeled_percentage = 0.2
num_labeled = int(len(y_train) * labeled_percentage)

X_labeled_view1 = X_train_view1[:num_labeled]
X_labeled_view2 = X_train_view2[:num_labeled]
y_labeled = y_train[:num_labeled]

X_unlabeled_view1 = X_train_view1[num_labeled:]
X_unlabeled_view2 = X_train_view2[num_labeled:]

# Initialize classifiers
clf1 = RandomForestClassifier(random_state=42)
# clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Fit on initial labeled data
clf1.fit(X_labeled_view1, y_labeled)
clf2.fit(X_labeled_view2, y_labeled)

# Co-training loop
max_iter = 10
samples_per_iter = 30  # number of unlabeled samples to label per iteration

for iteration in range(max_iter):
    # Predict probabilities on unlabeled data
    prob1 = clf1.predict_proba(X_unlabeled_view1)
    prob2 = clf2.predict_proba(X_unlabeled_view2)

    # Get most confident predictions
    confident_idx1 = np.argsort(-np.max(prob1, axis=1))[:samples_per_iter]
    confident_idx2 = np.argsort(-np.max(prob2, axis=1))[:samples_per_iter]

    # Select confident samples and pseudo-label them
    new_X1 = X_unlabeled_view1.iloc[confident_idx2]
    new_y1 = clf2.predict(X_unlabeled_view2.iloc[confident_idx2])

    new_X2 = X_unlabeled_view2.iloc[confident_idx1]
    new_y2 = clf1.predict(X_unlabeled_view1.iloc[confident_idx1])

    # Add to labeled data
    X_labeled_view1 = np.vstack([X_labeled_view1, new_X1])
    y_labeled = np.hstack([y_labeled, new_y1])

    X_labeled_view2 = np.vstack([X_labeled_view2, new_X2])

    # Remove labeled samples from unlabeled data
    mask1 = np.ones(len(X_unlabeled_view1), dtype=bool)
    mask1[confident_idx2] = False
    X_unlabeled_view1 = X_unlabeled_view1[mask1]
    X_unlabeled_view2 = X_unlabeled_view2[mask1]

    # Retrain classifiers
    clf1.fit(X_labeled_view1, y_labeled)
    clf2.fit(X_labeled_view2, y_labeled)

    if len(X_unlabeled_view1) < samples_per_iter:
        break

# Evaluate final model on validation set
val_pred1 = clf1.predict(X_val_view1)
val_pred2 = clf2.predict(X_val_view2)

print("Classifier 1 Validation Accuracy:", accuracy_score(y_val, val_pred1))
print("Classifier 1 Validation Report:\n", classification_report(y_val, val_pred1, digits=4))

print("Classifier 2 Validation Accuracy:", accuracy_score(y_val, val_pred2))
print("Classifier 2 Validation Report:\n", classification_report(y_val, val_pred2, digits=4))

# Test set performance
test_pred1 = clf1.predict(X_test_view1)
test_pred2 = clf2.predict(X_test_view2)

print("Classifier 1 Test Accuracy:", accuracy_score(y_test, test_pred1))
print("Classifier 1 Test Report:\n", classification_report(y_test, test_pred1, digits=4))

print("Classifier 2 Test Accuracy:", accuracy_score(y_test, test_pred2))
print("Classifier 2 Test Report:\n", classification_report(y_test, test_pred2, digits=4))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:54:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:54:27] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:54:33] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWar

Classifier 1 Validation Accuracy: 0.9785483870967742
Classifier 1 Validation Report:
               precision    recall  f1-score   support

           0     0.9730    0.9844    0.9787      9300
           1     0.9842    0.9727    0.9784      9300

    accuracy                         0.9785     18600
   macro avg     0.9786    0.9785    0.9785     18600
weighted avg     0.9786    0.9785    0.9785     18600

Classifier 2 Validation Accuracy: 0.94
Classifier 2 Validation Report:
               precision    recall  f1-score   support

           0     0.9293    0.9525    0.9407      9300
           1     0.9513    0.9275    0.9392      9300

    accuracy                         0.9400     18600
   macro avg     0.9403    0.9400    0.9400     18600
weighted avg     0.9403    0.9400    0.9400     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Classifier 1 Test Accuracy: 0.9758064516129032
Classifier 1 Test Report:
               precision    recall  f1-score   support

           0     0.9705    0.9814    0.9759      9300
           1     0.9812    0.9702    0.9757      9300

    accuracy                         0.9758     18600
   macro avg     0.9759    0.9758    0.9758     18600
weighted avg     0.9759    0.9758    0.9758     18600

Classifier 2 Test Accuracy: 0.942741935483871
Classifier 2 Test Report:
               precision    recall  f1-score   support

           0     0.9324    0.9547    0.9434      9300
           1     0.9536    0.9308    0.9420      9300

    accuracy                         0.9427     18600
   macro avg     0.9430    0.9427    0.9427     18600
weighted avg     0.9430    0.9427    0.9427     18600



**50% Labeled Data**

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Split feature set into two random views
from sklearn.model_selection import ShuffleSplit

splitter = ShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in splitter.split(X_train.T):
    view1_index, view2_index = train_index, test_index

X_train_view1 = X_train.iloc[:, view1_index]
X_train_view2 = X_train.iloc[:, view2_index]
X_val_view1 = X_val.iloc[:, view1_index]
X_val_view2 = X_val.iloc[:, view2_index]
X_test_view1 = X_test.iloc[:, view1_index]
X_test_view2 = X_test.iloc[:, view2_index]


# Split labeled and unlabeled data (20% labeled)
labeled_percentage = 0.5
num_labeled = int(len(y_train) * labeled_percentage)

X_labeled_view1 = X_train_view1[:num_labeled]
X_labeled_view2 = X_train_view2[:num_labeled]
y_labeled = y_train[:num_labeled]

X_unlabeled_view1 = X_train_view1[num_labeled:]
X_unlabeled_view2 = X_train_view2[num_labeled:]

# Initialize classifiers
clf1 = RandomForestClassifier(random_state=42)
# clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Fit on initial labeled data
clf1.fit(X_labeled_view1, y_labeled)
clf2.fit(X_labeled_view2, y_labeled)

# Co-training loop
max_iter = 10
samples_per_iter = 30  # number of unlabeled samples to label per iteration

for iteration in range(max_iter):
    # Predict probabilities on unlabeled data
    prob1 = clf1.predict_proba(X_unlabeled_view1)
    prob2 = clf2.predict_proba(X_unlabeled_view2)

    # Get most confident predictions
    confident_idx1 = np.argsort(-np.max(prob1, axis=1))[:samples_per_iter]
    confident_idx2 = np.argsort(-np.max(prob2, axis=1))[:samples_per_iter]

    # Select confident samples and pseudo-label them
    new_X1 = X_unlabeled_view1.iloc[confident_idx2]
    new_y1 = clf2.predict(X_unlabeled_view2.iloc[confident_idx2])

    new_X2 = X_unlabeled_view2.iloc[confident_idx1]
    new_y2 = clf1.predict(X_unlabeled_view1.iloc[confident_idx1])

    # Add to labeled data
    X_labeled_view1 = np.vstack([X_labeled_view1, new_X1])
    y_labeled = np.hstack([y_labeled, new_y1])

    X_labeled_view2 = np.vstack([X_labeled_view2, new_X2])

    # Remove labeled samples from unlabeled data
    mask1 = np.ones(len(X_unlabeled_view1), dtype=bool)
    mask1[confident_idx2] = False
    X_unlabeled_view1 = X_unlabeled_view1[mask1]
    X_unlabeled_view2 = X_unlabeled_view2[mask1]

    # Retrain classifiers
    clf1.fit(X_labeled_view1, y_labeled)
    clf2.fit(X_labeled_view2, y_labeled)

    if len(X_unlabeled_view1) < samples_per_iter:
        break

# Evaluate final model on validation set
val_pred1 = clf1.predict(X_val_view1)
val_pred2 = clf2.predict(X_val_view2)

print("Classifier 1 Validation Accuracy:", accuracy_score(y_val, val_pred1))
print("Classifier 1 Validation Report:\n", classification_report(y_val, val_pred1, digits=4))

print("Classifier 2 Validation Accuracy:", accuracy_score(y_val, val_pred2))
print("Classifier 2 Validation Report:\n", classification_report(y_val, val_pred2, digits=4))

# Test set performance
test_pred1 = clf1.predict(X_test_view1)
test_pred2 = clf2.predict(X_test_view2)

print("Classifier 1 Test Accuracy:", accuracy_score(y_test, test_pred1))
print("Classifier 1 Test Report:\n", classification_report(y_test, test_pred1, digits=4))

print("Classifier 2 Test Accuracy:", accuracy_score(y_test, test_pred2))
print("Classifier 2 Test Report:\n", classification_report(y_test, test_pred2, digits=4))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:58:09] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:58:22] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:58:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWar

Classifier 1 Validation Accuracy: 0.9847849462365591
Classifier 1 Validation Report:
               precision    recall  f1-score   support

           0     0.9802    0.9896    0.9849      9300
           1     0.9895    0.9800    0.9847      9300

    accuracy                         0.9848     18600
   macro avg     0.9848    0.9848    0.9848     18600
weighted avg     0.9848    0.9848    0.9848     18600

Classifier 2 Validation Accuracy: 0.9460752688172043
Classifier 2 Validation Report:
               precision    recall  f1-score   support

           0     0.9320    0.9624    0.9469      9300
           1     0.9611    0.9298    0.9452      9300

    accuracy                         0.9461     18600
   macro avg     0.9465    0.9461    0.9461     18600
weighted avg     0.9465    0.9461    0.9461     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Classifier 1 Test Accuracy: 0.9850537634408603
Classifier 1 Test Report:
               precision    recall  f1-score   support

           0     0.9803    0.9900    0.9851      9300
           1     0.9899    0.9801    0.9850      9300

    accuracy                         0.9851     18600
   macro avg     0.9851    0.9851    0.9851     18600
weighted avg     0.9851    0.9851    0.9851     18600

Classifier 2 Test Accuracy: 0.9489784946236559
Classifier 2 Test Report:
               precision    recall  f1-score   support

           0     0.9384    0.9611    0.9496      9300
           1     0.9601    0.9369    0.9484      9300

    accuracy                         0.9490     18600
   macro avg     0.9492    0.9490    0.9490     18600
weighted avg     0.9492    0.9490    0.9490     18600



**5% Labeled Data**

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Split feature set into two random views
from sklearn.model_selection import ShuffleSplit

splitter = ShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in splitter.split(X_train.T):
    view1_index, view2_index = train_index, test_index

X_train_view1 = X_train.iloc[:, view1_index]
X_train_view2 = X_train.iloc[:, view2_index]
X_val_view1 = X_val.iloc[:, view1_index]
X_val_view2 = X_val.iloc[:, view2_index]
X_test_view1 = X_test.iloc[:, view1_index]
X_test_view2 = X_test.iloc[:, view2_index]


# Split labeled and unlabeled data (20% labeled)
labeled_percentage = 0.05
num_labeled = int(len(y_train) * labeled_percentage)

X_labeled_view1 = X_train_view1[:num_labeled]
X_labeled_view2 = X_train_view2[:num_labeled]
y_labeled = y_train[:num_labeled]

X_unlabeled_view1 = X_train_view1[num_labeled:]
X_unlabeled_view2 = X_train_view2[num_labeled:]

# Initialize classifiers
clf1 = RandomForestClassifier(random_state=42)
# clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Fit on initial labeled data
clf1.fit(X_labeled_view1, y_labeled)
clf2.fit(X_labeled_view2, y_labeled)

# Co-training loop
max_iter = 10
samples_per_iter = 30  # number of unlabeled samples to label per iteration

for iteration in range(max_iter):
    # Predict probabilities on unlabeled data
    prob1 = clf1.predict_proba(X_unlabeled_view1)
    prob2 = clf2.predict_proba(X_unlabeled_view2)

    # Get most confident predictions
    confident_idx1 = np.argsort(-np.max(prob1, axis=1))[:samples_per_iter]
    confident_idx2 = np.argsort(-np.max(prob2, axis=1))[:samples_per_iter]

    # Select confident samples and pseudo-label them
    new_X1 = X_unlabeled_view1.iloc[confident_idx2]
    new_y1 = clf2.predict(X_unlabeled_view2.iloc[confident_idx2])

    new_X2 = X_unlabeled_view2.iloc[confident_idx1]
    new_y2 = clf1.predict(X_unlabeled_view1.iloc[confident_idx1])

    # Add to labeled data
    X_labeled_view1 = np.vstack([X_labeled_view1, new_X1])
    y_labeled = np.hstack([y_labeled, new_y1])

    X_labeled_view2 = np.vstack([X_labeled_view2, new_X2])

    # Remove labeled samples from unlabeled data
    mask1 = np.ones(len(X_unlabeled_view1), dtype=bool)
    mask1[confident_idx2] = False
    X_unlabeled_view1 = X_unlabeled_view1[mask1]
    X_unlabeled_view2 = X_unlabeled_view2[mask1]

    # Retrain classifiers
    clf1.fit(X_labeled_view1, y_labeled)
    clf2.fit(X_labeled_view2, y_labeled)

    if len(X_unlabeled_view1) < samples_per_iter:
        break

# Evaluate final model on validation set
val_pred1 = clf1.predict(X_val_view1)
val_pred2 = clf2.predict(X_val_view2)

print("Classifier 1 Validation Accuracy:", accuracy_score(y_val, val_pred1))
print("Classifier 1 Validation Report:\n", classification_report(y_val, val_pred1, digits=4))

print("Classifier 2 Validation Accuracy:", accuracy_score(y_val, val_pred2))
print("Classifier 2 Validation Report:\n", classification_report(y_val, val_pred2, digits=4))

# Test set performance
test_pred1 = clf1.predict(X_test_view1)
test_pred2 = clf2.predict(X_test_view2)

print("Classifier 1 Test Accuracy:", accuracy_score(y_test, test_pred1))
print("Classifier 1 Test Report:\n", classification_report(y_test, test_pred1, digits=4))

print("Classifier 2 Test Accuracy:", accuracy_score(y_test, test_pred2))
print("Classifier 2 Test Report:\n", classification_report(y_test, test_pred2, digits=4))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:00:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:00:50] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:00:53] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWar

Classifier 1 Validation Accuracy: 0.9655376344086022
Classifier 1 Validation Report:
               precision    recall  f1-score   support

           0     0.9618    0.9696    0.9657      9300
           1     0.9693    0.9615    0.9654      9300

    accuracy                         0.9655     18600
   macro avg     0.9656    0.9655    0.9655     18600
weighted avg     0.9656    0.9655    0.9655     18600

Classifier 2 Validation Accuracy: 0.9264516129032258
Classifier 2 Validation Report:
               precision    recall  f1-score   support

           0     0.9244    0.9288    0.9266      9300
           1     0.9285    0.9241    0.9263      9300

    accuracy                         0.9265     18600
   macro avg     0.9265    0.9265    0.9265     18600
weighted avg     0.9265    0.9265    0.9265     18600



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Classifier 1 Test Accuracy: 0.9648924731182795
Classifier 1 Test Report:
               precision    recall  f1-score   support

           0     0.9620    0.9681    0.9650      9300
           1     0.9679    0.9617    0.9648      9300

    accuracy                         0.9649     18600
   macro avg     0.9649    0.9649    0.9649     18600
weighted avg     0.9649    0.9649    0.9649     18600

Classifier 2 Test Accuracy: 0.9299462365591398
Classifier 2 Test Report:
               precision    recall  f1-score   support

           0     0.9284    0.9317    0.9301      9300
           1     0.9315    0.9282    0.9298      9300

    accuracy                         0.9299     18600
   macro avg     0.9300    0.9299    0.9299     18600
weighted avg     0.9300    0.9299    0.9299     18600



**10% Labeled Data**

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Split feature set into two random views
from sklearn.model_selection import ShuffleSplit

splitter = ShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in splitter.split(X_train.T):
    view1_index, view2_index = train_index, test_index

X_train_view1 = X_train.iloc[:, view1_index]
X_train_view2 = X_train.iloc[:, view2_index]
X_val_view1 = X_val.iloc[:, view1_index]
X_val_view2 = X_val.iloc[:, view2_index]
X_test_view1 = X_test.iloc[:, view1_index]
X_test_view2 = X_test.iloc[:, view2_index]


# Split labeled and unlabeled data (20% labeled)
labeled_percentage = 0.01
num_labeled = int(len(y_train) * labeled_percentage)

X_labeled_view1 = X_train_view1[:num_labeled]
X_labeled_view2 = X_train_view2[:num_labeled]
y_labeled = y_train[:num_labeled]

X_unlabeled_view1 = X_train_view1[num_labeled:]
X_unlabeled_view2 = X_train_view2[num_labeled:]

# Initialize classifiers
clf1 = RandomForestClassifier(random_state=42)
# clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Fit on initial labeled data
clf1.fit(X_labeled_view1, y_labeled)
clf2.fit(X_labeled_view2, y_labeled)

# Co-training loop
max_iter = 10
samples_per_iter = 30  # number of unlabeled samples to label per iteration

for iteration in range(max_iter):
    # Predict probabilities on unlabeled data
    prob1 = clf1.predict_proba(X_unlabeled_view1)
    prob2 = clf2.predict_proba(X_unlabeled_view2)

    # Get most confident predictions
    confident_idx1 = np.argsort(-np.max(prob1, axis=1))[:samples_per_iter]
    confident_idx2 = np.argsort(-np.max(prob2, axis=1))[:samples_per_iter]

    # Select confident samples and pseudo-label them
    new_X1 = X_unlabeled_view1.iloc[confident_idx2]
    new_y1 = clf2.predict(X_unlabeled_view2.iloc[confident_idx2])

    new_X2 = X_unlabeled_view2.iloc[confident_idx1]
    new_y2 = clf1.predict(X_unlabeled_view1.iloc[confident_idx1])

    # Add to labeled data
    X_labeled_view1 = np.vstack([X_labeled_view1, new_X1])
    y_labeled = np.hstack([y_labeled, new_y1])

    X_labeled_view2 = np.vstack([X_labeled_view2, new_X2])

    # Remove labeled samples from unlabeled data
    mask1 = np.ones(len(X_unlabeled_view1), dtype=bool)
    mask1[confident_idx2] = False
    X_unlabeled_view1 = X_unlabeled_view1[mask1]
    X_unlabeled_view2 = X_unlabeled_view2[mask1]

    # Retrain classifiers
    clf1.fit(X_labeled_view1, y_labeled)
    clf2.fit(X_labeled_view2, y_labeled)

    if len(X_unlabeled_view1) < samples_per_iter:
        break

# Evaluate final model on validation set
val_pred1 = clf1.predict(X_val_view1)
val_pred2 = clf2.predict(X_val_view2)

print("Classifier 1 Validation Accuracy:", accuracy_score(y_val, val_pred1))
print("Classifier 1 Validation Report:\n", classification_report(y_val, val_pred1, digits=4))

print("Classifier 2 Validation Accuracy:", accuracy_score(y_val, val_pred2))
print("Classifier 2 Validation Report:\n", classification_report(y_val, val_pred2, digits=4))

# Test set performance
test_pred1 = clf1.predict(X_test_view1)
test_pred2 = clf2.predict(X_test_view2)

print("Classifier 1 Test Accuracy:", accuracy_score(y_test, test_pred1))
print("Classifier 1 Test Report:\n", classification_report(y_test, test_pred1, digits=4))

print("Classifier 2 Test Accuracy:", accuracy_score(y_test, test_pred2))
print("Classifier 2 Test Report:\n", classification_report(y_test, test_pred2, digits=4))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:16] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:20] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWar

Classifier 1 Validation Accuracy: 0.944516129032258
Classifier 1 Validation Report:
               precision    recall  f1-score   support

           0     0.9476    0.9411    0.9443      9300
           1     0.9415    0.9480    0.9447      9300

    accuracy                         0.9445     18600
   macro avg     0.9445    0.9445    0.9445     18600
weighted avg     0.9445    0.9445    0.9445     18600

Classifier 2 Validation Accuracy: 0.8822043010752688
Classifier 2 Validation Report:
               precision    recall  f1-score   support

           0     0.8637    0.9076    0.8851      9300
           1     0.9027    0.8568    0.8791      9300

    accuracy                         0.8822     18600
   macro avg     0.8832    0.8822    0.8821     18600
weighted avg     0.8832    0.8822    0.8821     18600

Classifier 1 Test Accuracy: 0.9447311827956989
Classifier 1 Test Report:
               precision    recall  f1-score   support

           0     0.9489    0.9401    0.9445   

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


**1% Labeled Data**

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Split feature set into two random views
from sklearn.model_selection import ShuffleSplit

splitter = ShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for train_index, test_index in splitter.split(X_train.T):
    view1_index, view2_index = train_index, test_index

X_train_view1 = X_train.iloc[:, view1_index]
X_train_view2 = X_train.iloc[:, view2_index]
X_val_view1 = X_val.iloc[:, view1_index]
X_val_view2 = X_val.iloc[:, view2_index]
X_test_view1 = X_test.iloc[:, view1_index]
X_test_view2 = X_test.iloc[:, view2_index]


# Split labeled and unlabeled data (20% labeled)
labeled_percentage = 0.01
num_labeled = int(len(y_train) * labeled_percentage)

X_labeled_view1 = X_train_view1[:num_labeled]
X_labeled_view2 = X_train_view2[:num_labeled]
y_labeled = y_train[:num_labeled]

X_unlabeled_view1 = X_train_view1[num_labeled:]
X_unlabeled_view2 = X_train_view2[num_labeled:]

# Initialize classifiers
clf1 = RandomForestClassifier(random_state=42)
# clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Fit on initial labeled data
clf1.fit(X_labeled_view1, y_labeled)
clf2.fit(X_labeled_view2, y_labeled)

# Co-training loop
max_iter = 10
samples_per_iter = 30  # number of unlabeled samples to label per iteration

for iteration in range(max_iter):
    # Predict probabilities on unlabeled data
    prob1 = clf1.predict_proba(X_unlabeled_view1)
    prob2 = clf2.predict_proba(X_unlabeled_view2)

    # Get most confident predictions
    confident_idx1 = np.argsort(-np.max(prob1, axis=1))[:samples_per_iter]
    confident_idx2 = np.argsort(-np.max(prob2, axis=1))[:samples_per_iter]

    # Select confident samples and pseudo-label them
    new_X1 = X_unlabeled_view1.iloc[confident_idx2]
    new_y1 = clf2.predict(X_unlabeled_view2.iloc[confident_idx2])

    new_X2 = X_unlabeled_view2.iloc[confident_idx1]
    new_y2 = clf1.predict(X_unlabeled_view1.iloc[confident_idx1])

    # Add to labeled data
    X_labeled_view1 = np.vstack([X_labeled_view1, new_X1])
    y_labeled = np.hstack([y_labeled, new_y1])

    X_labeled_view2 = np.vstack([X_labeled_view2, new_X2])

    # Remove labeled samples from unlabeled data
    mask1 = np.ones(len(X_unlabeled_view1), dtype=bool)
    mask1[confident_idx2] = False
    X_unlabeled_view1 = X_unlabeled_view1[mask1]
    X_unlabeled_view2 = X_unlabeled_view2[mask1]

    # Retrain classifiers
    clf1.fit(X_labeled_view1, y_labeled)
    clf2.fit(X_labeled_view2, y_labeled)

    if len(X_unlabeled_view1) < samples_per_iter:
        break

# Evaluate final model on validation set
val_pred1 = clf1.predict(X_val_view1)
val_pred2 = clf2.predict(X_val_view2)

print("Classifier 1 Validation Accuracy:", accuracy_score(y_val, val_pred1))
print("Classifier 1 Validation Report:\n", classification_report(y_val, val_pred1, digits=4))

print("Classifier 2 Validation Accuracy:", accuracy_score(y_val, val_pred2))
print("Classifier 2 Validation Report:\n", classification_report(y_val, val_pred2, digits=4))

# Test set performance
test_pred1 = clf1.predict(X_test_view1)
test_pred2 = clf2.predict(X_test_view2)

print("Classifier 1 Test Accuracy:", accuracy_score(y_test, test_pred1))
print("Classifier 1 Test Report:\n", classification_report(y_test, test_pred1, digits=4))

print("Classifier 2 Test Accuracy:", accuracy_score(y_test, test_pred2))
print("Classifier 2 Test Report:\n", classification_report(y_test, test_pred2, digits=4))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:35] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:37] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:39] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWar

Classifier 1 Validation Accuracy: 0.944516129032258
Classifier 1 Validation Report:
               precision    recall  f1-score   support

           0     0.9476    0.9411    0.9443      9300
           1     0.9415    0.9480    0.9447      9300

    accuracy                         0.9445     18600
   macro avg     0.9445    0.9445    0.9445     18600
weighted avg     0.9445    0.9445    0.9445     18600

Classifier 2 Validation Accuracy: 0.8822043010752688
Classifier 2 Validation Report:
               precision    recall  f1-score   support

           0     0.8637    0.9076    0.8851      9300
           1     0.9027    0.8568    0.8791      9300

    accuracy                         0.8822     18600
   macro avg     0.8832    0.8822    0.8821     18600
weighted avg     0.8832    0.8822    0.8821     18600

Classifier 1 Test Accuracy: 0.9447311827956989
Classifier 1 Test Report:
               precision    recall  f1-score   support

           0     0.9489    0.9401    0.9445   

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
